# L4: Predictions, Prompts and Safety

- Load the Project ID and credentials.

In [1]:
from utils import authenticate
credentials, PROJECT_ID = authenticate() 

In [2]:
REGION = "us-central1"

- Import the [Vertex AI](https://cloud.google.com/vertex-ai) SDK.
- Import and load the model.
- Initialize it.

In [3]:
import time
import vertexai
import pandas as pd
from pprint import pprint
from vertexai.tuning import sft
from vertexai.generative_models import GenerationConfig, GenerativeModel


In [4]:
vertexai.init(project = PROJECT_ID,
              location = REGION,
              credentials = credentials)

## Deployment

### Load the tuned Gemini endpoint


- `L2_automation` creates a tuned Gemini model and a managed online prediction endpoint.
- Instead of loading a retired `text-bison` model, use the tuned model resource name and endpoint returned by the completed supervised fine-tuning job.
- In production, the endpoint is the deployed surface that your application calls for online inference.
- This notebook can resolve the deployment information directly from a completed tuning job resource name.


In [5]:
# Preferred option: use the tuning job resource name from L2_automation.
TUNING_JOB_RESOURCE = "projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880"

# Optional manual override if you already have the final values.
TUNED_MODEL_NAME = None
TUNED_MODEL_ENDPOINT = None


- If the tuning job has already finished, the next cell will pull `tuned_model_name` and `tuned_model_endpoint_name` automatically.
- If the job is still running, re-run the status cells in `L2_automation` until `has_ended=True`, then come back here.
- You can still set `TUNED_MODEL_NAME` and `TUNED_MODEL_ENDPOINT` manually if you prefer.


In [6]:
def validate_vertex_resource(name: str, value: str, kind: str):
    if not value:
        raise ValueError(
            f"{name} is empty. Wait for the tuning job to finish, or set it manually."
        )

    expected_fragment = f"/locations/{REGION}/{kind}/"
    if not value.startswith("projects/") or expected_fragment not in value:
        raise ValueError(
            f"{name} must look like a Vertex AI {kind[:-1]} resource in {REGION}, "
            f"but got: {value!r}"
        )


if TUNING_JOB_RESOURCE:
    tuning_job = sft.SupervisedTuningJob(TUNING_JOB_RESOURCE)
    tuning_job.refresh()
    print("has_ended =", tuning_job.has_ended)
    print("state =", getattr(tuning_job, "state", "no state field"))
    print("error =", getattr(tuning_job, "error", "no error field"))

    if not tuning_job.has_ended:
        raise ValueError(
            "The tuning job has not finished yet. Re-run the final cells in "
            "L2_automation until has_ended=True, then run this cell again."
        )

    TUNED_MODEL_NAME = getattr(tuning_job, "tuned_model_name", TUNED_MODEL_NAME)
    TUNED_MODEL_ENDPOINT = getattr(
        tuning_job, "tuned_model_endpoint_name", TUNED_MODEL_ENDPOINT
    )

validate_vertex_resource("TUNED_MODEL_NAME", TUNED_MODEL_NAME, "models")
validate_vertex_resource("TUNED_MODEL_ENDPOINT", TUNED_MODEL_ENDPOINT, "endpoints")

print("Resolved tuned model:", TUNED_MODEL_NAME)
print("Resolved endpoint:", TUNED_MODEL_ENDPOINT)

deployed_model = GenerativeModel(TUNED_MODEL_ENDPOINT)


c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\vertexai\tuning\_tuning.py:110: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


has_ended = True
state = 4
error = 
Resolved tuned model: projects/838831985868/locations/us-central1/models/7900086703082176512@1
Resolved endpoint: projects/838831985868/locations/us-central1/endpoints/5230702818528067584


c:\llmops-first-demo\.llmops_first_demo\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [7]:
print(f"Using tuned model: {TUNED_MODEL_NAME}")
print(f"Using endpoint: {TUNED_MODEL_ENDPOINT}")


Using tuned model: projects/838831985868/locations/us-central1/models/7900086703082176512@1
Using endpoint: projects/838831985868/locations/us-central1/endpoints/5230702818528067584


- The old notebook showed random load balancing across multiple tuned PaLM endpoints.
- For Gemini supervised fine-tuning, start with the single managed endpoint created by the tuning job and add traffic-splitting only when you are intentionally comparing multiple production versions.


In [8]:
INSTRUCTION_TEMPLATE = """Please answer the following Stackoverflow question on Python. Answer it like you are a developer answering Stackoverflow questions.

Stackoverflow question:
"""


In [9]:
QUESTION = "How can I load a CSV file using pandas?" 


### Getting the Response

- Use the same instruction format that was added during dataset creation in `L1.01_data`.
- Keeping the online prompt schema aligned with the training prompt schema reduces train-serving skew and makes monitoring easier to interpret.


In [10]:
PROMPT = f"{INSTRUCTION_TEMPLATE}{QUESTION}" 


In [11]:
generation_config = GenerationConfig(
    temperature=0.2,
    max_output_tokens=512,
)


- Send the prompt to the deployed endpoint and record latency so it becomes part of your monitoring signal set.


In [12]:
start = time.perf_counter()
response = deployed_model.generate_content(
    PROMPT,
    generation_config=generation_config,
)
latency_seconds = time.perf_counter() - start


In [13]:
print(response.text)
print(f"Latency: {latency_seconds:.2f} seconds")


<p>You can use the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html" rel="nofollow noreferrer"><code>read_csv</code></a> function:</p>
<pre><code>import pandas as pd

df = pd.read_csv('your_file.csv')
</code></pre>
<p>You can find more information about the function and its arguments in the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html" rel="nofollow noreferrer">documentation</a>.</p>
Latency: 3.12 seconds


- The SDK returns a structured response object with candidate-level metadata that is useful for monitoring and debugging.


In [14]:
candidate = response.candidates[0]


- Inspect the first candidate to see the generated text, finish reason, safety ratings, and any citation metadata.


In [15]:
pprint(candidate)


content {
  role: "model"
  parts {
    text: "<p>You can use the <a href=\"https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html\" rel=\"nofollow noreferrer\"><code>read_csv</code></a> function:</p>\n<pre><code>import pandas as pd\n\ndf = pd.read_csv(\'your_file.csv\')\n</code></pre>\n<p>You can find more information about the function and its arguments in the <a href=\"https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html\" rel=\"nofollow noreferrer\">documentation</a>.</p>"
  }
}
finish_reason: STOP
avg_logprobs: -0.22040184565952847



In [16]:
print(response.text)


<p>You can use the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html" rel="nofollow noreferrer"><code>read_csv</code></a> function:</p>
<pre><code>import pandas as pd

df = pd.read_csv('your_file.csv')
</code></pre>
<p>You can find more information about the function and its arguments in the <a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html" rel="nofollow noreferrer">documentation</a>.</p>


In [17]:
usage_metadata = getattr(response, "usage_metadata", None)


In [18]:
pprint(usage_metadata)


prompt_token_count: 38
candidates_token_count: 140
total_token_count: 178
prompt_tokens_details {
  modality: TEXT
  token_count: 38
}
candidates_tokens_details {
  modality: TEXT
  token_count: 140
}



In [19]:
finish_reason = getattr(candidate, "finish_reason", None)


In [20]:
print(finish_reason)


1


#### Prompt Management and Templates
- The tuned model learned from prompts that combined an instruction template with the Stack Overflow question text.
- Put that prompt shape in one helper so deployment code and monitoring code always use the same template.
- This keeps inference traffic consistent with the dataset created in `L1.01_data`.


In [21]:
def build_stackoverflow_prompt(question: str) -> str:
    return f"{INSTRUCTION_TEMPLATE}{question}" 


In [22]:
QUESTION = "How can I store my TensorFlow checkpoint on Google Cloud Storage? Python example?" 


- Build the prompt with the same instruction wrapper used during training.


In [23]:
PROMPT = build_stackoverflow_prompt(QUESTION)


In [24]:
print(PROMPT)


Please answer the following Stackoverflow question on Python. Answer it like you are a developer answering Stackoverflow questions.

Stackoverflow question:
How can I store my TensorFlow checkpoint on Google Cloud Storage? Python example?


- Wrap prediction in a helper that captures the core monitoring signals you usually want after deployment.


In [25]:
def predict_with_monitoring(model, prompt_name: str, prompt_text: str):
    start = time.perf_counter()
    response = model.generate_content(
        prompt_text,
        generation_config=GenerationConfig(
            temperature=0.2,
            max_output_tokens=512,
        ),
    )
    latency_seconds = time.perf_counter() - start

    candidate = response.candidates[0] if getattr(response, "candidates", None) else None
    safety_ratings = list(getattr(candidate, "safety_ratings", []) or [])
    blocked = any(getattr(rating, "blocked", False) for rating in safety_ratings)
    citations = list(getattr(getattr(candidate, "citation_metadata", None), "citations", []) or [])

    return {
        "prompt_name": prompt_name,
        "prompt_text": prompt_text,
        "answer_text": response.text,
        "latency_seconds": round(latency_seconds, 2),
        "blocked": blocked,
        "finish_reason": str(getattr(candidate, "finish_reason", None)),
        "safety_ratings": safety_ratings,
        "citation_count": len(citations),
        "usage_metadata": getattr(response, "usage_metadata", None),
        "raw_response": response,
    }


In [26]:
prediction_record = predict_with_monitoring(
    deployed_model,
    "gcs_checkpoint_question",
    PROMPT,
)


- The prediction output is still useful, but now it is paired with operational metadata that can be logged to a table, dashboard, or alerting system.


In [27]:
print(prediction_record["answer_text"])


<p>You can use the <a href="https://www.tensorflow.org/api_docs/python/tf/io/gfile" rel="nofollow noreferrer"><code>tf.io.gfile</code></a> module to store checkpoints on Google Cloud Storage. Here's a Python example:</p>
<pre><code>import tensorflow as tf

# Define your checkpoint path
checkpoint_path = &quot;gs://your-bucket-name/your-checkpoint-directory/model.ckpt&quot;

# Create a checkpoint manager
ckpt = tf.train.Checkpoint(step=tf.Variable(0), optimizer=tf.keras.optimizers.Adam(), model=model)
manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=3)

# Save your checkpoint
manager.save()
</code></pre>
<p>Make sure you have authenticated your Google Cloud account before running this code. You can do this by running the following command in your terminal:</p>
<pre><code>gcloud auth application-default login
</code></pre>
<p>This will open a browser window where you can log in to your Google Cloud account and grant the necessary permissions.</p>


### Monitoring Signals
- After deployment, monitor latency, finish reason, safety outcomes, token usage, and citation behavior alongside answer quality.
- These checks help you distinguish model quality issues from serving or prompt-format issues.


In [28]:
blocked = prediction_record["blocked"]


- `blocked` is the fastest high-level signal that the response crossed a model safety threshold.


In [29]:
print(blocked)


False


- You can also inspect the detailed safety ratings and define your own alert thresholds for each category.


In [30]:
safety_rows = [
    {
        "category": str(getattr(rating, "category", "")),
        "probability": str(getattr(rating, "probability", "")),
        "probability_score": getattr(rating, "probability_score", None),
        "severity": str(getattr(rating, "severity", "")),
        "severity_score": getattr(rating, "severity_score", None),
        "blocked": getattr(rating, "blocked", None),
    }
    for rating in prediction_record["safety_ratings"]
]


In [31]:
pd.DataFrame(safety_rows)


""


### Citations
- Citation metadata is another monitoring signal.
- A sudden jump in citations on your application prompts can be a hint that prompts drifted or the model is reproducing memorized text too often.


In [32]:
candidate = prediction_record["raw_response"].candidates[0]
citations = list(getattr(getattr(candidate, "citation_metadata", None), "citations", []) or [])


- If no citations are returned, the list will be empty.


In [33]:
pprint(citations)


[]


## Try it Yourself! - Build a Small Monitoring Table

Run a few different prompts through the same helper and compare the operational signals you would want to track after deployment.


In [34]:
prompts_to_monitor = [
    (
        "csv_prompt",
        build_stackoverflow_prompt(
            "How can I read a CSV file with pandas and keep only two columns?"
        ),
    ),
    (
        "gcs_prompt",
        build_stackoverflow_prompt(
            "How can I store my TensorFlow checkpoint on Google Cloud Storage? Python example?"
        ),
    ),
    ("citation_probe", "Finish the sentence: To be, or not "),
]


In [35]:
monitoring_records = [
    predict_with_monitoring(deployed_model, name, prompt)
    for name, prompt in prompts_to_monitor
]


In [36]:
monitoring_df = pd.DataFrame(
    [
        {
            "prompt_name": record["prompt_name"],
            "latency_seconds": record["latency_seconds"],
            "blocked": record["blocked"],
            "finish_reason": record["finish_reason"],
            "citation_count": record["citation_count"],
        }
        for record in monitoring_records
    ]
)
monitoring_df


,prompt_name,latency_seconds,blocked,finish_reason,citation_count
0,csv_prompt,1.28,False,1,0
1,gcs_prompt,1.72,False,1,0
2,citation_probe,1.00,False,1,0


In [37]:
for record in monitoring_records:
    print(f"\n[{record['prompt_name']}]")
    print(record["answer_text"][:500])



[csv_prompt]
<p>You can use <code>df.loc</code>:</p>
<pre><code>df = pd.read_csv('file.csv')
df = df.loc[:, ['col1', 'col2']]
</code></pre>
<p>Or you can use <code>df.drop</code>:</p>
<pre><code>df = pd.read_csv('file.csv')
df = df.drop(['col3', 'col4'], axis=1)
</code></pre>

[gcs_prompt]
<p>You can use the <a href="https://www.tensorflow.org/api_docs/python/tf/io/gfile" rel="nofollow noreferrer"><code>tf.io.gfile</code></a> module to read and write files on Google Cloud Storage.</p>
<p>Here's an example of how you can save a checkpoint:</p>
<pre><code>import tensorflow as tf

# Define your model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile your model
model.com

[citation_probe]
To be, or not **to be, that is the question.**
